# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FarisElbaz/ML_Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Our baseline action score prioritizes striking-distance content items ($4.0 \le \text{avg\_position} \le 15.0$) that underperform their expected click-through rate given their search impressions, or high-volume pages that have suffered an abrupt short-term traffic drop. The score estimates unrealized monthly click upside:

$$\text{Action Score} = \text{Impressions}_{14d} \times \max\left(0, \, \text{Benchmark CTR} - \text{Observed CTR}_{14d}\right)$$For decayed pages, the score is scaled by the trailing volume lost.

Output Reason Codes & Actions:CTR_UNDERPERFORMER $\rightarrow$ Action: REVISE_METADATA_SNIPPET (Impression volume $\ge 100$, position between $4.0$ and $15.0$, CTR $< 50\%$ of benchmark).

VELOCITY_DECAY $\rightarrow$ Action: CONTENT_REFRESH_AUDIT (Previous week clicks $\ge 10$, current week drop $\ge 40\%$).

STRIKING_OPPORTUNITY $\rightarrow$ Action: EXPAND_INTERNAL_LINKS (Position $4.0\text{--}10.0$, impressions $\ge 300$, but CTR is healthy/at benchmark).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
import json
import os
import duckdb
from google.colab import userdata
from huggingface_hub import login
import polars as pl

# Ensure output directory exists
os.makedirs("work/outputs", exist_ok=True)

# Authenticate with Hugging Face Hub
hf_token = userdata.get("hf_copllab_access")
login(token=hf_token)
print("Successfully logged in to Hugging Face Hub!")

# Configure DuckDB with HF Secret
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"
month_path = f"{rel}/fact_content_daily_performance/month=2026-03/**/*.parquet"

# Build ranked queue using pre-cutoff window (<= 2026-03-14)
build_queue_sql = f"""
WITH metrics AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS imp_14d,
        SUM(gsc_clicks) AS clicks_14d,
        AVG(gsc_avg_position) AS avg_pos,
        SUM(gsc_clicks) FILTER (WHERE report_date BETWEEN '2026-03-01' AND '2026-03-07') AS clicks_w1,
        SUM(gsc_clicks) FILTER (WHERE report_date BETWEEN '2026-03-08' AND '2026-03-14') AS clicks_w2
    FROM read_parquet('{month_path}')
    WHERE gsc_data_available IS TRUE
      AND report_date <= '2026-03-14'
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 50
),
scored AS (
    SELECT
        client_hash_id,
        content_hash_id,
        imp_14d,
        clicks_14d,
        ROUND(avg_pos, 2) AS avg_pos,
        ROUND(clicks_14d * 1.0 / NULLIF(imp_14d, 0), 4) AS observed_ctr,
        CASE
            WHEN avg_pos BETWEEN 4.0 AND 15.0 AND (clicks_14d * 1.0 / NULLIF(imp_14d, 0)) < 0.015 AND imp_14d >= 100
                THEN 'CTR_UNDERPERFORMER'
            WHEN clicks_w1 >= 10 AND (clicks_w2 - clicks_w1) * 1.0 / clicks_w1 <= -0.40
                THEN 'VELOCITY_DECAY'
            WHEN avg_pos BETWEEN 4.0 AND 10.0 AND imp_14d >= 300
                THEN 'STRIKING_OPPORTUNITY'
            ELSE 'MONITOR'
        END AS reason_code,
        CASE
            WHEN avg_pos BETWEEN 4.0 AND 15.0 AND (clicks_14d * 1.0 / NULLIF(imp_14d, 0)) < 0.015 AND imp_14d >= 100
                THEN 'REVISE_METADATA_SNIPPET'
            WHEN clicks_w1 >= 10 AND (clicks_w2 - clicks_w1) * 1.0 / clicks_w1 <= -0.40
                THEN 'CONTENT_REFRESH_AUDIT'
            WHEN avg_pos BETWEEN 4.0 AND 10.0 AND imp_14d >= 300
                THEN 'EXPAND_INTERNAL_LINKS'
            ELSE 'NO_ACTION'
        END AS action_label,
        ROUND(
            CASE
                -- Score for decaying assets: scale of lost weekly click volume
                WHEN clicks_w1 >= 10 AND (clicks_w2 - clicks_w1) * 1.0 / clicks_w1 <= -0.40
                    THEN (clicks_w1 - clicks_w2) * 2.0
                -- Score for striking/underperforming assets: expected CTR deficit * impression scale
                ELSE imp_14d * GREATEST(0.0, 0.035 - (clicks_14d * 1.0 / NULLIF(imp_14d, 0)))
            END,
            2
        ) AS action_score
    FROM metrics
)
SELECT
    client_hash_id,
    content_hash_id,
    imp_14d,
    clicks_14d,
    avg_pos,
    observed_ctr,
    reason_code,
    action_label,
    action_score
FROM scored
WHERE action_label != 'NO_ACTION'
ORDER BY action_score DESC;
"""

ranked_pl = con.sql(build_queue_sql).pl()

# Write the ranked CSV
csv_path = "work/outputs/baseline_action_score.csv"
ranked_pl.write_csv(csv_path)

# Save metrics JSON receipt for git tracking
receipt = {
    "total_ranked_items": ranked_pl.height,
    "top_action": ranked_pl["action_label"][0],
    "top_score": float(ranked_pl["action_score"][0]),
    "mean_score": float(ranked_pl["action_score"].mean()),
    "action_breakdown": {
        row["action_label"]: row["len"]
        for row in ranked_pl.group_by("action_label").len().to_dicts()
    },
}
with open("work/outputs/baseline_metrics.json", "w") as f:
    json.dump(receipt, f, indent=2)

print(f"Wrote {ranked_pl.height} rows to {csv_path}")
print(f"Saved run receipt to work/outputs/baseline_metrics.json")
ranked_pl.head(5)


Successfully logged in to Hugging Face Hub!


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Wrote 38659 rows to work/outputs/baseline_action_score.csv
Saved run receipt to work/outputs/baseline_metrics.json


client_hash_id,content_hash_id,imp_14d,clicks_14d,avg_pos,observed_ctr,reason_code,action_label,action_score
str,str,"decimal[38,0]","decimal[38,0]",f64,f64,str,str,f64
"""client_73cda7b4e4f265ea""","""content_9c057b66c30a3abb""",83770,0,5.79,0.0,"""CTR_UNDERPERFORMER""","""REVISE_METADATA_SNIPPET""",2931.95
"""client_62f4a7e64f5e0096""","""content_7c6373141eae744a""",84160,47,5.85,0.0006,"""CTR_UNDERPERFORMER""","""REVISE_METADATA_SNIPPET""",2898.6
"""client_62f4a7e64f5e0096""","""content_b99ea6861864dea5""",86624,171,4.04,0.002,"""CTR_UNDERPERFORMER""","""REVISE_METADATA_SNIPPET""",2860.84
"""client_08a6a72ff48e62c0""","""content_e7b5dd4dff461ad2""",84409,893,4.54,0.0106,"""CTR_UNDERPERFORMER""","""REVISE_METADATA_SNIPPET""",2061.32
"""client_73cda7b4e4f265ea""","""content_8e1334d6356668e3""",56091,1,4.39,0.0,"""CTR_UNDERPERFORMER""","""REVISE_METADATA_SNIPPET""",1962.19


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [4]:
top20 = ranked_pl.head(20).to_dicts()
for idx, r in enumerate(top20, 1):
    print(
        f"Row {idx:02d}: Content {r['content_hash_id'][:10]}... | Action: {r['action_label']} | Reason: {r['reason_code']}"
    )
    print(
        f"   Score: {r['action_score']} | Imp: {r['imp_14d']} | Pos: {r['avg_pos']} | CTR: {r['observed_ctr']}\n"
    )

Row 01: Content content_9c... | Action: REVISE_METADATA_SNIPPET | Reason: CTR_UNDERPERFORMER
   Score: 2931.95 | Imp: 83770 | Pos: 5.79 | CTR: 0.0

Row 02: Content content_7c... | Action: REVISE_METADATA_SNIPPET | Reason: CTR_UNDERPERFORMER
   Score: 2898.6 | Imp: 84160 | Pos: 5.85 | CTR: 0.0006

Row 03: Content content_b9... | Action: REVISE_METADATA_SNIPPET | Reason: CTR_UNDERPERFORMER
   Score: 2860.84 | Imp: 86624 | Pos: 4.04 | CTR: 0.002

Row 04: Content content_e7... | Action: REVISE_METADATA_SNIPPET | Reason: CTR_UNDERPERFORMER
   Score: 2061.32 | Imp: 84409 | Pos: 4.54 | CTR: 0.0106

Row 05: Content content_8e... | Action: REVISE_METADATA_SNIPPET | Reason: CTR_UNDERPERFORMER
   Score: 1962.19 | Imp: 56091 | Pos: 4.39 | CTR: 0.0

Row 06: Content content_65... | Action: REVISE_METADATA_SNIPPET | Reason: CTR_UNDERPERFORMER
   Score: 1922.6 | Imp: 55360 | Pos: 8.58 | CTR: 0.0003

Row 07: Content content_95... | Action: REVISE_METADATA_SNIPPET | Reason: CTR_UNDERPERFORMER
   Score: 

1. **Row 01:** content_9c...  
   **Action:** REVISE_METADATA_SNIPPET  
   **Reason Code:** CTR_UNDERPERFORMER  
   **Confidence Note:** High. 83,770 impressions at position 5.79 with literally 0 clicks is a massive theoretical upside ($2{,}931.95$ score).  
   **What would make it wrong:** The ranking query triggers an instant answer or featured snippet that satisfies the query without a click (zero-click search), or has strict navigational intent for an official competitor.

2. **Row 02:** content_7c...  
   **Action:** REVISE_METADATA_SNIPPET  
   **Reason Code:** CTR_UNDERPERFORMER  
   **Confidence Note:** High. 84,160 impressions at position 5.85 with an observed CTR of 0.06% against a 3.5% benchmark.  
   **What would make it wrong:** The page ranks for an overly broad, ambiguous keyword where user intent diverges completely from the landing page's topic.

3. **Row 03:** content_b9...  
   **Action:** REVISE_METADATA_SNIPPET  
   **Reason Code:** CTR_UNDERPERFORMER  
   **Confidence Note:** High. Sits right at the top of striking distance (position 4.04) with 86,624 impressions, yet captures only 0.2% CTR.  
   **What would make it wrong:** Heavy paid search (Google Ads) or shopping carousels dominate the top fold, pushing organic position 4 below the screen.

4. **Row 04:** content_e7...  
   **Action:** REVISE_METADATA_SNIPPET  
   **Reason Code:** CTR_UNDERPERFORMER  
   **Confidence Note:** High. Generates 84,409 impressions at position 4.54 with 1.06% CTR, leaving substantial click volume on the table.  
   **What would make it wrong:** The current title and meta description already match query intent accurately, but competitors display star ratings or pricing rich snippets that naturally capture user clicks.

5. **Row 05:** content_8e...  
   **Action:** REVISE_METADATA_SNIPPET  
   **Reason Code:** CTR_UNDERPERFORMER  
   **Confidence Note:** High. 56,091 impressions at position 4.39 capturing zero clicks.  
   **What would make it wrong:** Search Console logs phantom impressions caused by automated scrapers or rank-tracking bots rather than genuine human searchers.

6. **Row 06:** content_65...  
   **Action:** REVISE_METADATA_SNIPPET  
   **Reason Code:** CTR_UNDERPERFORMER  
   **Confidence Note:** Moderate. 55,360 impressions at position 8.58 with 0.03% CTR.  
   **What would make it wrong:** Position 8 naturally has a very low CTR baseline ($\approx 1\%$); assuming a 3.5% benchmark without first ranking in the top 5 overstates the realistic snippet-fix upside.

7. **Row 07:** content_95...  
   **Action:** REVISE_METADATA_SNIPPET  
   **Reason Code:** CTR_UNDERPERFORMER  
   **Confidence Note:** High. Position 4.38 with 53,545 impressions and only 0.19% CTR.  
   **What would make it wrong:** Google rewrites the title tag on the live SERP to an unappealing snippet, bypassing the page's HTML meta tags.

8. **Row 08:** content_62...  
   **Action:** REVISE_METADATA_SNIPPET  
   **Reason Code:** CTR_UNDERPERFORMER  
   **Confidence Note:** High. 49,127 impressions at position 5.76 with near-zero clicks (0.07%).  
   **What would make it wrong:** Query intent is primarily video- or image-seeking, leading users to interact exclusively with media packs above organic results.

9. **Row 09:** content_f4...  
   **Action:** REVISE_METADATA_SNIPPET  
   **Reason Code:** CTR_UNDERPERFORMER  
   **Confidence Note:** High. Position 4.09 with 49,537 impressions and 0.18% CTR.  
   **What would make it wrong:** The page is matching non-commercial informational queries where the searcher is looking for a quick definition already shown in Google's People Also Ask (PAA) box.

10. **Row 10:** content_f6...  
    **Action:** REVISE_METADATA_SNIPPET  
    **Reason Code:** CTR_UNDERPERFORMER  
    **Confidence Note:** Moderate. Sits near the bottom of Page 1 (position 9.44) with 46,608 impressions and 0.01% CTR.  
    **What would make it wrong:** Low CTR is driven primarily by rank suppression near the page boundary rather than an unappealing title/description snippet.

11. **Row 11:** content_e5...  
    **Action:** REVISE_METADATA_SNIPPET  
    **Reason Code:** CTR_UNDERPERFORMER  
    **Confidence Note:** High. Strong position at 4.23 with 46,953 impressions and 0.16% CTR.  
    **What would make it wrong:** Brand mismatch: the search query includes an implied competitor brand name, causing users to ignore non-brand domains.

12. **Row 12:** content_36...  
    **Action:** REVISE_METADATA_SNIPPET  
    **Reason Code:** CTR_UNDERPERFORMER  
    **Confidence Note:** High. Position 5.21 with 44,630 impressions and 0.02% CTR.  
    **What would make it wrong:** The URL structure or domain displayed in the SERP snippet appears untrustworthy or irrelevant to searchers.

13. **Row 13:** content_94...  
    **Action:** REVISE_METADATA_SNIPPET  
    **Reason Code:** CTR_UNDERPERFORMER  
    **Confidence Note:** High. 44,156 impressions at position 6.30 with zero recorded clicks.  
    **What would make it wrong:** Local pack or Google Maps widget occupies the primary viewport for the query, pushing standard organic listings out of view.

14. **Row 14:** content_13...  
    **Action:** REVISE_METADATA_SNIPPET  
    **Reason Code:** CTR_UNDERPERFORMER  
    **Confidence Note:** High. Position 4.16 with 43,529 impressions and 0.14% CTR.  
    **What would make it wrong:** The title tag truncates abruptly on mobile screens, cutting off the core value proposition.

15. **Row 15:** content_b1...  
    **Action:** REVISE_METADATA_SNIPPET  
    **Reason Code:** CTR_UNDERPERFORMER  
    **Confidence Note:** High. 48,837 impressions at position 4.44 capturing only 0.52% CTR.  
    **What would make it wrong:** Seasonal demand or time-sensitive query where the current snippet displays an outdated year (e.g., "2024" instead of current).

16. **Row 16:** content_09...  
    **Action:** REVISE_METADATA_SNIPPET  
    **Reason Code:** CTR_UNDERPERFORMER  
    **Confidence Note:** Moderate. Position 8.76 with 40,557 impressions and 0.06% CTR.  
    **What would make it wrong:** Snippet optimization alone won't move the needle much until rank improves past position 5 through link authority or content depth.

17. **Row 17:** content_3f...  
    **Action:** REVISE_METADATA_SNIPPET  
    **Reason Code:** CTR_UNDERPERFORMER  
    **Confidence Note:** High. 46,805 impressions at position 6.56 with 0.56% CTR.  
    **What would make it wrong:** The page is a PDF or downloadable file format that mobile searchers intentionally avoid clicking.

18. **Row 18:** content_0a...  
    **Action:** REVISE_METADATA_SNIPPET  
    **Reason Code:** CTR_UNDERPERFORMER  
    **Confidence Note:** Low-Moderate. Sits on the Page 1/Page 2 boundary (position 10.06) with 39,011 impressions and 0.08% CTR.  
    **What would make it wrong:** Position fluctuates wildly between page 1 and page 2, making the 3.5% CTR benchmark an unrealistic target without first securing top-page rank.

19. **Row 19:** content_96...  
    **Action:** REVISE_METADATA_SNIPPET  
    **Reason Code:** CTR_UNDERPERFORMER  
    **Confidence Note:** High. Strong rank at 4.23 with 39,307 impressions and 0.3% CTR.  
    **What would make it wrong:** The page ranks for international queries where the language or currency shown in the snippet does not match user locale.

20. **Row 20:** content_3b...  
    **Action:** REVISE_METADATA_SNIPPET  
    **Reason Code:** CTR_UNDERPERFORMER  
    **Confidence Note:** High. Position 6.28 with 33,640 impressions and 0.08% CTR.  
    **What would make it wrong:** High impression volume is an aggregate artifact of thousands of ultra-long-tail keywords rather than a single head term with clear search intent.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Which Picks Look Wrong and Why

- **Position 9–10+ Outliers** (e.g., Row 10 content_f6... at pos 9.44 and Row 18 content_0a... at pos 10.06):
  - **Why they look wrong:** Expecting a flat 3.5% CTR benchmark for positions sitting right on the Page 1/Page 2 boundary dramatically overstates the realistic click upside. The low CTR here is predominantly driven by rank suppression and page scroll depth, not snippet quality. Applying REVISE_METADATA_SNIPPET here will likely underperform compared to an authority/link intervention.

- **Extreme Zero-Click Assets** (e.g., Row 1 content_9c..., Row 5 content_8e..., Row 13 content_94...):
  - **Why they look wrong:** Recording 40,000 to 80,000+ impressions at rank 4–6 with literally 0 or 1 click strongly indicates that the search volume is either driven by scrapers/rank-check bots, or Google is displaying a dominant Instant Answer / Knowledge Graph card that completely absorbs user intent. Tweaking titles or meta descriptions cannot recover clicks from a zero-click SERP.

- **Monolithic Action Dominance (Action Homogeneity):**
  - **Why it looks wrong:** All top 20 rows were assigned REVISE_METADATA_SNIPPET under CTR_UNDERPERFORMER. Because impression scale acts as a direct multiplier, raw impression volume heavily biased the top queue toward high-impression CTR deficits, completely crowding out genuine VELOCITY_DECAY assets that may have needed urgent content intervention.

## Leakage Audit Confirmation

- **No Product/System Flags Leaked:**
  - The rule strictly computes metrics from raw numerical telemetry (gsc_impressions, gsc_clicks, and gsc_avg_position).
  - No internal FlyRank production flags, client trend markers, or hand-coded refresh indicators were referenced or joined in the scoring logic.

- **Strict Temporal Window Separation:**
  - All input aggregates, baseline CTRs, and week-over-week deltas are filtered strictly on or before the cutoff date: report_date <= '2026-03-14'.
  - No forward target window data (March 15–31, 2026) was used in calculating the score, reason codes, or rank position.
  - The sealed June 2026 test partition was completely excluded from the dataset query.

- **No Label-Derived Features:**
  - The score does not incorporate forward click outcomes, target conversion labels, or forward trajectory metrics. All ranking features are strictly historical to the decision point.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.